# Stage 4 — ID Type Classifier training (EfficientNet-B0)

**Before running this notebook**, make sure you have:
1. Run `colab_00_setup.ipynb` in this same Colab session (mounts Drive, clones repo, extracts data).
2. The zip includes `data/id_type/all/<class>/` (produced by `convert_labels_to_yolo.py --id-type` on your work PC).
3. The Stage 1 model weights are at `models/stage1_corners/weights/best.pt` (included in the zip via `sync_to_cloud.ps1 --include-models`, or download separately with `sync_from_cloud.ps1 -Stage stage1`).

**Workflow in this notebook:**
1. Deskew all 782 id_type images through Stage 1 → `data/id_type/all_deskewed/`
2. Split deskewed images into train/val/test
3. Train EfficientNet-B0 on the deskewed splits
4. Save weights to Google Drive

Classes: `legacy`, `maisha`, `huduma`, `passport`, `driving_licence`, `foreign_document`, `unknown_id`

In [ ]:
EPOCHS = 50
BATCH  = 32
DEVICE = "cuda"  # use 'cpu' to test locally

In [ ]:
import sys
sys.path.insert(0, "/content/id-forensics-model/scripts")
import colab_bootstrap as cb

cb.check_gpu()

In [ ]:
# Step 1: Verify raw id_type images are present
from pathlib import Path

all_root = Path("/content/id-forensics-model/data/id_type/all")
classes = ["legacy", "maisha", "huduma", "passport", "driving_licence", "foreign_document", "unknown_id"]
total = 0
for cls in classes:
    n = len(list((all_root / cls).glob("*.jpg"))) if (all_root / cls).is_dir() else 0
    print(f"  {cls:20s}: {n}")
    total += n
print(f"\nTotal raw images: {total}")
if total == 0:
    print("\nWARNING: No images found in data/id_type/all/")
    print("On work PC: python scripts/convert_labels_to_yolo.py --id-type")
    print("Then re-pack with: .\\scripts\\sync_to_cloud.ps1 and re-extract in colab_00_setup.")

In [ ]:
# Step 2: Deskew all id_type images through Stage 1
# This runs id_crop.run() on every image and saves corrected crops.
# Partial / low-confidence images are saved as-is (fallback).
!cd /content/id-forensics-model && python scripts/deskew_id_type_images.py

In [ ]:
# Step 3: Stratified train/val/test split from deskewed images
!cd /content/id-forensics-model && python scripts/split_id_type_dataset.py --source all_deskewed

In [ ]:
!cd /content/id-forensics-model && python scripts/training/train_stage4_id_type.py \
    --device {DEVICE} --epochs {EPOCHS} --batch {BATCH}

In [ ]:
import sys
import importlib
sys.path.insert(0, "/content/id-forensics-model/scripts")
import colab_bootstrap as cb
importlib.reload(cb)  # force reload in case of cached older version

cb.save_weights("stage4")